# COVID-19 Patient Sera Preprocessing

This notebook prepares the peptide-variant intensity matrix for classification models. The final model-ready table has one patient/sample per row, peptide variants as features, and an encoded condition label.

## 1. Load libraries and data

Import the Python packages needed for preprocessing, then load the MAESTRO TSV file into a pandas dataframe.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)


/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [2]:
DATA_PATH = Path(
    "data/ProteoSAFe-MAESTRO-d6178bdd-identified_variants_merged_protein_regions/"
    "MAESTRO-d6178bdd-identified_variants_merged_protein_regions-main.tsv"
)

raw_df = pd.read_csv(DATA_PATH, sep="\t")
raw_df.shape


/var/folders/jm/zbjslnk917g737fvb65lwd300000gn/T/ipykernel_94380/3309052869.py:6: DtypeWarning: Columns (259) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(DATA_PATH, sep="\t")


(101461, 268)

In [3]:
raw_df.head()


,rowid,ccms_row_id,Algorithm,Filename,Cluster_index,Peptide,Unmodified_sequence,Charge,_dyn_#Intensity_for_cluster,_dyn_#Intensity_for_unmodified_sequence,...,PSP_site_match,DrugBank_drugs,Parent_mass,Num_PSP_Drugbank_events,Start_AA_1_based,End_AA_1_based,Num_spectra_for_cluster,Num_spectra_for_unmodified_sequence,Num_spectra_for_peptide_variant,Internal_ref_orig_intensity
0,1,1,.MODA.,specs_ms.mgf,960991,"K.[304.207]GARLIPEMDQIFTEVEMTTLE(K,304.207).V",.GARLIPEMDQIFTEVEMTTLEK.,4,36.905893,36.905893,...,NaN,NaN,1580.81,0,NaN,NaN,1,1,1,8.204159e+03
1,2,2,.MODA.,specs_ms.mgf,763982,"I.[304.207]FTEVEMTTLE(K,304.207).V",.FTEVEMTTLEK.,3,11.686782,11.686782,...,NaN,NaN,1934.91,0,NaN,NaN,1,2,2,4.936894e+05
2,3,3,.MSGFPLUS.,specs_ms.mgf,902201,K.[304.207]LYQPEYQEVSTEEQR.E,.LYQPEYQEVSTEEQR.,3,15.690234,15.690234,...,NaN,NaN,2203.09,0,NaN,NaN,5,6,6,1.951566e+05
3,4,4,.MSGFPLUS.,specs_ms.mgf,935503,"K.[304.207]AANSLEAFIFETQD(K,304.207).L",.AANSLEAFIFETQDK.,3,15.016824,15.016824,...,NaN,NaN,2292.24,0,NaN,NaN,3,4,4,2.877781e+06
4,5,5,.MODA.,specs_ms.mgf,297961,"R.[304.207]YSHDF(N,-56.985)FH.I",.YSHDFNFH.,3,33.768015,33.768015,...,NaN,NaN,1313.66,0,NaN,NaN,3,3,3,7.088440e+04


## 2. Keep peptide-variant intensity columns

Select the `Peptide` column and the intensity columns ending in `_intensity_for_peptide_variant`, which are the measurements used as model features.

In [4]:
INTENSITY_SUFFIX = "_intensity_for_peptide_variant"

intensity_cols = [col for col in raw_df.columns if col.endswith(INTENSITY_SUFFIX)]
peptide_df = raw_df[["Peptide", *intensity_cols]].copy()

print(f"Peptide rows: {peptide_df.shape[0]:,}")
print(f"Peptide-variant intensity columns: {len(intensity_cols):,}")
peptide_df.head()


Peptide rows: 101,461
Peptide-variant intensity columns: 92


,Peptide,_dyn_#Empty.Empty.Empty..Empty.1_intensity_for_peptide_variant,_dyn_#Healthy.HC1.Healthy..HC1.1_intensity_for_peptide_variant,_dyn_#Healthy.HC10.Healthy..HC10.1_intensity_for_peptide_variant,_dyn_#Healthy.HC12.Healthy..HC12.1_intensity_for_peptide_variant,_dyn_#Healthy.HC13.Healthy..HC13.1_intensity_for_peptide_variant,_dyn_#Healthy.HC17.Healthy..HC17.1_intensity_for_peptide_variant,_dyn_#Healthy.HC19.Healthy..HC19.1_intensity_for_peptide_variant,_dyn_#Healthy.HC2.Healthy..HC2.1_intensity_for_peptide_variant,_dyn_#Healthy.HC20.Healthy..HC20.1_intensity_for_peptide_variant,...,_dyn_#Symptomatic-non-COVID-19.JBDZ24.Symptomatic-non-COVID-19..JBDZ24.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ25.Symptomatic-non-COVID-19..JBDZ25.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ3.Symptomatic-non-COVID-19..JBDZ3.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ4.Symptomatic-non-COVID-19..JBDZ4.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ5.Symptomatic-non-COVID-19..JBDZ5.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ6.Symptomatic-non-COVID-19..JBDZ6.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ7.Symptomatic-non-COVID-19..JBDZ7.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ8.Symptomatic-non-COVID-19..JBDZ8.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.JBDZ9.Symptomatic-non-COVID-19..JBDZ9.1_intensity_for_peptide_variant,_dyn_#Symptomatic-non-COVID-19.Patient-group-jbdz.Symptomatic-non-COVID-19..Patient-group-jbdz.1_intensity_for_peptide_variant
0,"K.[304.207]GARLIPEMDQIFTEVEMTTLE(K,304.207).V",2.459416,0.000000,6.645649,3.391896,1.919552,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.000000
1,"I.[304.207]FTEVEMTTLE(K,304.207).V",0.961707,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.435434,0.736280,0.79655,1.144616
2,K.[304.207]LYQPEYQEVSTEEQR.E,0.326271,0.935916,0.000000,0.000000,0.000000,0.286530,0.177772,0.987496,0.220930,...,0.000000,0.000000,1.232362,0.0,0.0,0.0,0.200232,0.170478,0.19740,0.047076
3,"K.[304.207]AANSLEAFIFETQD(K,304.207).L",0.878024,0.000000,0.000000,0.000000,0.000000,1.259306,0.629756,0.000000,0.781082,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.000000
4,"R.[304.207]YSHDF(N,-56.985)FH.I",0.170619,0.000000,0.000000,0.000000,0.000000,0.000000,0.238642,0.000000,0.272645,...,0.796323,0.832859,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.000000


## 3. Parse condition and sample ID

Extract patient condition and sample ID from each intensity column name so labels and metadata are explicit.

In [5]:
def parse_intensity_column(col: str) -> dict:
    """Extract condition and sample_id from a MAESTRO intensity column name."""
    prefix = col.removeprefix("_dyn_#").removesuffix(INTENSITY_SUFFIX)
    parts = prefix.split(".")
    return {
        "original_column": col,
        "condition": parts[0],
        "sample_id": parts[1],
    }


sample_metadata = pd.DataFrame([parse_intensity_column(col) for col in intensity_cols])
sample_metadata["condition"].value_counts()


condition
Non-severe-COVID-19         25
Symptomatic-non-COVID-19    25
Healthy                     22
Severe-COVID-19             18
Empty                        1
Norm                         1
Name: count, dtype: int64

In [6]:
sample_metadata.head()


,original_column,condition,sample_id
0,_dyn_#Empty.Empty.Empty..Empty.1_intensity_for...,Empty,Empty
1,_dyn_#Healthy.HC1.Healthy..HC1.1_intensity_for...,Healthy,HC1
2,_dyn_#Healthy.HC10.Healthy..HC10.1_intensity_f...,Healthy,HC10
3,_dyn_#Healthy.HC12.Healthy..HC12.1_intensity_f...,Healthy,HC12
4,_dyn_#Healthy.HC13.Healthy..HC13.1_intensity_f...,Healthy,HC13


## 4. Remove non-patient controls

Drop `Empty` and `Norm` columns because they are controls rather than patient observations.

In [7]:
CONTROL_CONDITIONS = {"Empty", "Norm"}

patient_metadata = sample_metadata[~sample_metadata["condition"].isin(CONTROL_CONDITIONS)].copy()
patient_cols = patient_metadata["original_column"].tolist()

patient_peptide_df = peptide_df[["Peptide", *patient_cols]].copy()

print(f"Patient intensity columns: {len(patient_cols):,}")
patient_metadata["condition"].value_counts()


Patient intensity columns: 90


condition
Non-severe-COVID-19         25
Symptomatic-non-COVID-19    25
Healthy                     22
Severe-COVID-19             18
Name: count, dtype: int64

## 5. Replace zero intensities with missing values

Convert zero intensity values to `NaN` before filtering, since zeros in MS data usually represent non-detection.

In [8]:
intensity_matrix = patient_peptide_df.set_index("Peptide")

intensity_matrix = intensity_matrix.apply(pd.to_numeric, errors="coerce")

zero_count = (intensity_matrix == 0).sum().sum()
intensity_matrix = intensity_matrix.replace(0, np.nan)

print(f"Zero intensities converted to NaN: {zero_count:,}")
print(f"Total missing values after zero replacement: {intensity_matrix.isna().sum().sum():,}")


Zero intensities converted to NaN: 5,363,463
Total missing values after zero replacement: 5,410,263


## 6. Remove highly missing peptides

Filter out peptides missing in more than 50% of patient samples to reduce sparse and unreliable features.

In [9]:
MISSINGNESS_THRESHOLD = 0.50

missing_fraction_by_peptide = intensity_matrix.isna().mean(axis=1)
keep_peptides = missing_fraction_by_peptide <= MISSINGNESS_THRESHOLD

filtered_matrix = intensity_matrix.loc[keep_peptides].copy()

print(f"Peptides before filtering: {intensity_matrix.shape[0]:,}")
print(f"Peptides after filtering: {filtered_matrix.shape[0]:,}")
print(f"Peptides removed: {(~keep_peptides).sum():,}")


Peptides before filtering: 101,461
Peptides after filtering: 35,882
Peptides removed: 65,579


## 7. Impute and transform intensities

Fill remaining missing values with `0` to represent non-detection, then apply `log1p` to reduce intensity skew.

In [10]:
imputed_matrix = filtered_matrix.fillna(0)

log_matrix = np.log1p(imputed_matrix)

print(f"Remaining missing values: {log_matrix.isna().sum().sum():,}")
log_matrix.iloc[:5, :5]


Remaining missing values: 0


,_dyn_#Healthy.HC1.Healthy..HC1.1_intensity_for_peptide_variant,_dyn_#Healthy.HC10.Healthy..HC10.1_intensity_for_peptide_variant,_dyn_#Healthy.HC12.Healthy..HC12.1_intensity_for_peptide_variant,_dyn_#Healthy.HC13.Healthy..HC13.1_intensity_for_peptide_variant,_dyn_#Healthy.HC17.Healthy..HC17.1_intensity_for_peptide_variant
Peptide,,,,,
K.[304.207]LYQPEYQEVSTEEQR.E,0.660581,0.000000,0.000000,0.000000,0.251949
"K.[304.207]H(G,304.213)TDDGVVWMNW(K,304.207).G",0.253728,0.025691,0.008992,0.028019,0.000000
"R.[304.207]DDTV(C,57.021)LA(K,304.207)LHDR.N",0.000000,0.000000,0.000000,0.000000,0.891504
"R.[304.207]QQQHLFGSNVTD(C,57.021)SGNF(C,57.021)LFR.S",0.096477,0.191623,0.090532,0.120605,0.099547
"-.[304.207](Q,-17.027)QQHLFGSNVTD(C,57.021)SGNF(C,57.021)LFR.S",0.841409,0.631683,0.328496,0.435484,0.000000


## 8. Transpose to samples x peptides

Transpose the matrix so each row is a patient/sample and each column is a peptide feature.

In [11]:
sample_feature_matrix = log_matrix.T
sample_feature_matrix.index.name = "original_column"

model_df = sample_feature_matrix.merge(
    patient_metadata.set_index("original_column")[["condition", "sample_id"]],
    left_index=True,
    right_index=True,
)

print(model_df.shape)
model_df.head()


(90, 35884)


,K.[304.207]LYQPEYQEVSTEEQR.E,"K.[304.207]H(G,304.213)TDDGVVWMNW(K,304.207).G","R.[304.207]DDTV(C,57.021)LA(K,304.207)LHDR.N","R.[304.207]QQQHLFGSNVTD(C,57.021)SGNF(C,57.021)LFR.S","-.[304.207](Q,-17.027)QQHLFGSNVTD(C,57.021)SGNF(C,57.021)LFR.S","K.[304.207]DLLFRDDTV(C,57.021)L(A,304.58)(K,304.207).L","R.[304.207]DDTV(C,57.021)LA(K,632.887)LHDR.N","R.[304.207]QQQHL(F,305.275)GSNVTD(C,57.021)SGNF(C,57.021)LFR.S","R.[304.207]QQQHLF(G,304.219)SNVTD(C,57.021)SGNF(C,57.021)LFR.S","R.[304.207]DDTV(C,57.021)(L,21.982)A(K,304.207).L",...,"K.[304.207]YLGE(E,31.996)YV(K,304.207).A","K.[304.207]YLGEE(Y,21.985)V(K,304.207).A","K.[304.207]YLGE(E,14.998)YV(K,304.207).A","K.[304.207]YL(G,13.987)EEYV(K,304.207).A","K.[304.207]YLGEEYV(K,302.195).A","K.[304.207]YLGEEY(V,-18.002)(K,304.207).A","K.[304.207]YLGEE(Y,-20.114)V(K,304.207).A","K.[304.207]YLGEE(Y,-57.005)V(K,304.207).A",condition,sample_id
original_column,,,,,,,,,,,,,,,,,,,,,
_dyn_#Healthy.HC1.Healthy..HC1.1_intensity_for_peptide_variant,0.660581,0.253728,0.000000,0.096477,0.841409,0.000000,0.000000,0.000000,0.855375,0.000000,...,0.244423,0.654389,0.000000,0.187094,0.357646,0.338294,0.454854,1.084792,Healthy,HC1
_dyn_#Healthy.HC10.Healthy..HC10.1_intensity_for_peptide_variant,0.000000,0.025691,0.000000,0.191623,0.631683,0.309955,0.000000,0.550178,0.312313,0.323958,...,0.000000,0.908099,0.078225,0.259012,0.732251,0.309250,0.329653,1.265440,Healthy,HC10
_dyn_#Healthy.HC12.Healthy..HC12.1_intensity_for_peptide_variant,0.000000,0.008992,0.000000,0.090532,0.328496,0.171289,0.000000,0.318865,0.161258,0.187374,...,0.000000,0.533243,0.048059,0.138023,0.430190,0.155832,0.179494,0.648777,Healthy,HC12
_dyn_#Healthy.HC13.Healthy..HC13.1_intensity_for_peptide_variant,0.000000,0.028019,0.000000,0.120605,0.435484,0.276920,0.000000,0.438251,0.249076,0.191860,...,0.000000,0.652852,0.055279,0.196770,0.357765,0.205598,0.225932,0.710716,Healthy,HC13
_dyn_#Healthy.HC17.Healthy..HC17.1_intensity_for_peptide_variant,0.251949,0.000000,0.891504,0.099547,0.000000,0.462195,0.301303,0.432989,0.272124,0.000000,...,0.000000,0.551745,0.718191,0.212386,0.000000,0.384190,0.435244,0.105163,Healthy,HC17


## 9. Create task-specific classification datasets

Define the three binary tasks from the project report. Each task reuses the same peptide feature columns, but includes a different subset of samples and gets its own labels.


In [12]:
TASK_CONFIGS = {
    "healthy_vs_infected": {
        "positive_conditions": [
            "Non-severe-COVID-19",
            "Severe-COVID-19",
            "Symptomatic-non-COVID-19",
        ],
        "negative_conditions": ["Healthy"],
        "positive_label": "Infected",
        "negative_label": "Healthy",
    },
    "symptomatic_non_covid_vs_covid": {
        "positive_conditions": ["Non-severe-COVID-19", "Severe-COVID-19"],
        "negative_conditions": ["Symptomatic-non-COVID-19"],
        "positive_label": "COVID-19",
        "negative_label": "Symptomatic-non-COVID-19",
    },
    "severe_vs_nonsevere": {
        "positive_conditions": ["Severe-COVID-19"],
        "negative_conditions": ["Non-severe-COVID-19"],
        "positive_label": "Severe-COVID-19",
        "negative_label": "Non-severe-COVID-19",
    },
}

feature_cols = [col for col in model_df.columns if col not in {"condition", "sample_id"}]

for task_name, config in TASK_CONFIGS.items():
    task_conditions = config["negative_conditions"] + config["positive_conditions"]
    task_df = model_df[model_df["condition"].isin(task_conditions)].copy()
    task_df["label"] = task_df["condition"].isin(config["positive_conditions"]).astype(int)
    task_df["label_name"] = np.where(
        task_df["label"].eq(1),
        config["positive_label"],
        config["negative_label"],
    )

    print(f"\n{task_name}")
    print(task_df["condition"].value_counts())
    print(task_df["label_name"].value_counts())



healthy_vs_infected
condition
Non-severe-COVID-19         25
Symptomatic-non-COVID-19    25
Healthy                     22
Severe-COVID-19             18
Name: count, dtype: int64
label_name
Infected    68
Healthy     22
Name: count, dtype: int64

symptomatic_non_covid_vs_covid
condition
Non-severe-COVID-19         25
Symptomatic-non-COVID-19    25
Severe-COVID-19             18
Name: count, dtype: int64
label_name
COVID-19                    43
Symptomatic-non-COVID-19    25
Name: count, dtype: int64

severe_vs_nonsevere
condition
Non-severe-COVID-19    25
Severe-COVID-19        18
Name: count, dtype: int64
label_name
Non-severe-COVID-19    25
Severe-COVID-19        18
Name: count, dtype: int64


In [13]:
def make_task_split(task_name, config, test_size=0.2, random_state=42):
    task_conditions = config["negative_conditions"] + config["positive_conditions"]
    task_df = model_df[model_df["condition"].isin(task_conditions)].copy()
    task_df["label"] = task_df["condition"].isin(config["positive_conditions"]).astype(int)
    task_df["label_name"] = np.where(
        task_df["label"].eq(1),
        config["positive_label"],
        config["negative_label"],
    )

    X = task_df[feature_cols].copy()
    y = task_df["label"].copy()
    metadata = task_df[["sample_id", "condition", "label", "label_name"]].copy()

    X_train, X_test, y_train, y_test, metadata_train, metadata_test = train_test_split(
        X,
        y,
        metadata,
        test_size=test_size,
        random_state=random_state,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        index=X_train.index,
        columns=X_train.columns,
    )
    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        index=X_test.index,
        columns=X_test.columns,
    )

    return {
        "task_name": task_name,
        "config": config,
        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "metadata_train": metadata_train,
        "metadata_test": metadata_test,
        "n_features": len(feature_cols),
    }


task_splits = {
    task_name: make_task_split(task_name, config)
    for task_name, config in TASK_CONFIGS.items()
}

for task_name, split in task_splits.items():
    print(f"\n{task_name}")
    print(f"Train shape: {split['X_train_scaled'].shape}")
    print(f"Test shape: {split['X_test_scaled'].shape}")
    print("Train labels:")
    print(split["metadata_train"]["label_name"].value_counts())
    print("Test labels:")
    print(split["metadata_test"]["label_name"].value_counts())



healthy_vs_infected
Train shape: (72, 35882)
Test shape: (18, 35882)
Train labels:
label_name
Infected    54
Healthy     18
Name: count, dtype: int64
Test labels:
label_name
Infected    14
Healthy      4
Name: count, dtype: int64

symptomatic_non_covid_vs_covid
Train shape: (54, 35882)
Test shape: (14, 35882)
Train labels:
label_name
COVID-19                    34
Symptomatic-non-COVID-19    20
Name: count, dtype: int64
Test labels:
label_name
COVID-19                    9
Symptomatic-non-COVID-19    5
Name: count, dtype: int64

severe_vs_nonsevere
Train shape: (34, 35882)
Test shape: (9, 35882)
Train labels:
label_name
Non-severe-COVID-19    20
Severe-COVID-19        14
Name: count, dtype: int64
Test labels:
label_name
Non-severe-COVID-19    5
Severe-COVID-19        4
Name: count, dtype: int64


## 10. Train/test split and scaling

Split each task with stratification and fit scaling only on the training set to avoid test-data leakage.


In [14]:
summary_rows = []

for task_name, split in task_splits.items():
    summary_rows.append({
        "task": task_name,
        "train_samples": split["X_train_scaled"].shape[0],
        "test_samples": split["X_test_scaled"].shape[0],
        "features": split["n_features"],
        "train_label_counts": split["metadata_train"]["label_name"].value_counts().to_dict(),
        "test_label_counts": split["metadata_test"]["label_name"].value_counts().to_dict(),
    })

split_summary_df = pd.DataFrame(summary_rows)
split_summary_df


,task,train_samples,test_samples,features,train_label_counts,test_label_counts
0,healthy_vs_infected,72,18,35882,"{'Infected': 54, 'Healthy': 18}","{'Infected': 14, 'Healthy': 4}"
1,symptomatic_non_covid_vs_covid,54,14,35882,"{'COVID-19': 34, 'Symptomatic-non-COVID-19': 20}","{'COVID-19': 9, 'Symptomatic-non-COVID-19': 5}"
2,severe_vs_nonsevere,34,9,35882,"{'Non-severe-COVID-19': 20, 'Severe-COVID-19':...","{'Non-severe-COVID-19': 5, 'Severe-COVID-19': 4}"


## 11. Save processed outputs

Write each task's processed feature matrices, labels, metadata, and selected peptide list to a task-specific subfolder in `data/processed`.


In [15]:
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for task_name, split in task_splits.items():
    task_dir = OUTPUT_DIR / task_name
    task_dir.mkdir(parents=True, exist_ok=True)

    split["X_train_scaled"].to_csv(task_dir / "X_train_scaled.csv")
    split["X_test_scaled"].to_csv(task_dir / "X_test_scaled.csv")
    split["y_train"].to_csv(task_dir / "y_train.csv", header=True)
    split["y_test"].to_csv(task_dir / "y_test.csv", header=True)
    split["metadata_train"].to_csv(task_dir / "metadata_train.csv")
    split["metadata_test"].to_csv(task_dir / "metadata_test.csv")
    pd.Series(feature_cols, name="peptide").to_csv(task_dir / "selected_peptides.csv", index=False)

split_summary_df.to_csv(OUTPUT_DIR / "task_split_summary.csv", index=False)
pd.Series(feature_cols, name="peptide").to_csv(OUTPUT_DIR / "selected_peptides.csv", index=False)

print(f"Saved task-specific processed files to: {OUTPUT_DIR.resolve()}")


Saved task-specific processed files to: /Users/krithikaiyer/cse291-project/data/processed
